In [2]:
!pip install -q transformers accelerate sentencepiece

In [3]:
!pip install networkx

In [4]:
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.eval()

print("Qwen 1.5B loaded successfully")
print("GPU available:", torch.cuda.is_available())

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen 1.5B loaded successfully
GPU available: False


In [15]:
def llm_extract(text):

    messages = [
        {
            "role": "system",
            "content": """You are a strict knowledge extraction engine.

Your task is to convert ANY natural-language scenario into structured JSON.

IMPORTANT:
1. Extract EVERY factual statement.
2. Extract EVERY condition in EVERY rule.
3. Do NOT invent facts.
4. Do NOT change operators.
5. "at least" means >=.
6. "at most" means <=.
7. "more than" means >.
8. "less than" means <.
9. "no X" means X == false.
10. "has X" or "completed X" means X == true when used as a condition.
11. Preserve the actual value from the facts.
12. The final decision must be determined by the symbolic engine, NOT by you.

Return ONLY valid JSON.
No explanation.
No markdown.
No text before or after JSON.

Use EXACTLY this structure:

{
  "facts": [
    {
      "entity": "Arun",
      "attribute": "credit_score",
      "value": 750
    }
  ],
  "rules": [
    {
      "conditions": [
        {
          "attribute": "credit_score",
          "operator": ">=",
          "value": 700
        }
      ],
      "conclusion": {
        "attribute": "eligible",
        "value": true
      }
    }
  ]
}

Rules:
- Every condition MUST contain attribute, operator, and value.
- Every conclusion MUST contain attribute and value.
- Every fact MUST contain entity, attribute, and value.
- Use one condition for each requirement.
- Do not combine multiple conditions into one condition.
- Do not put conditions inside facts.
- Do not put facts inside rules.
- Use simple attribute names such as credit_score, monthly_income, gpa, attendance, experience.
- Use boolean true/false for yes/no conditions.
- Return a complete JSON object."""
        },
        {
            "role": "user",
            "content": text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return response

In [16]:
def clean_llm_json(output):

    output = output.strip()

    output = re.sub(
        r"```json|```",
        "",
        output,
        flags=re.IGNORECASE
    )

    start = output.find("{")

    if start == -1:
        raise ValueError("No JSON object found.")

    depth = 0
    end = -1

    for i in range(start, len(output)):

        if output[i] == "{":
            depth += 1

        elif output[i] == "}":
            depth -= 1

            if depth == 0:
                end = i + 1
                break

    if end == -1:
        raise ValueError("Incomplete JSON.")

    json_text = output[start:end]

    return json.loads(json_text)

In [17]:
def normalize_value(value):

    if isinstance(value, str):

        value = value.strip()

        if value.lower() == "true":
            return True

        if value.lower() == "false":
            return False

        try:

            number = float(
                value.replace("%", "")
            )

            if number.is_integer():
                return int(number)

            return number

        except:
            return value

    return value


def normalize_knowledge(data):

    for fact in data.get("facts", []):

        fact["value"] = normalize_value(
            fact["value"]
        )

    for rule in data.get("rules", []):

        for condition in rule["conditions"]:

            condition["value"] = normalize_value(
                condition["value"]
            )

        rule["conclusion"]["value"] = normalize_value(
            rule["conclusion"]["value"]
        )

    return data

In [18]:

import networkx as nx

def build_knowledge_graph(data):
    graph = nx.DiGraph()

    for fact_id, fact in enumerate(data.get("facts", [])):
        entity = str(fact.get("entity", "Unknown"))
        attribute = str(fact.get("attribute", "Unknown"))
        value = fact.get("value")

        entity_node = f"entity:{entity}"
        fact_node = f"fact:{fact_id}"
        value_node = f"value:{fact_id}"

        graph.add_node(entity_node, node_type="entity", name=entity)
        graph.add_node(fact_node, node_type="fact", attribute=attribute)
        graph.add_node(value_node, node_type="value", value=value)

        graph.add_edge(entity_node, fact_node, relation="HAS_FACT")
        graph.add_edge(fact_node, value_node, relation="HAS_VALUE")

    for rule_id, rule in enumerate(data.get("rules", [])):
        rule_node = f"rule:{rule_id}"
        graph.add_node(rule_node, node_type="rule")

        for condition_id, condition in enumerate(rule.get("conditions", [])):
            condition_node = f"condition:{rule_id}:{condition_id}"

            graph.add_node(
                condition_node,
                node_type="condition",
                attribute=condition.get("attribute"),
                operator=condition.get("operator"),
                value=condition.get("value")
            )

            graph.add_edge(rule_node, condition_node, relation="HAS_CONDITION")

        conclusion = rule.get("conclusion", {})

        conclusion_node = f"conclusion:{rule_id}"
        graph.add_node(
            conclusion_node,
            node_type="conclusion",
            attribute=conclusion.get("attribute"),
            value=conclusion.get("value")
        )

        graph.add_edge(rule_node, conclusion_node, relation="CONCLUDES")

    return graph


def get_graph_fact(graph, entity, attribute):
    entity_node = f"entity:{entity}"

    if entity_node not in graph:
        return None

    for _, fact_node, edge_data in graph.out_edges(entity_node, data=True):
        if edge_data.get("relation") != "HAS_FACT":
            continue

        if graph.nodes[fact_node].get("attribute") != attribute:
            continue

        for _, value_node, value_edge in graph.out_edges(fact_node, data=True):
            if value_edge.get("relation") == "HAS_VALUE":
                return graph.nodes[value_node].get("value")

    return None


def print_knowledge_graph(graph):
    print("\nKNOWLEDGE GRAPH")
    print("-" * 60)

    for source, target, attributes in graph.edges(data=True):
        print(
            f"{source} --[{attributes['relation']}]--> {target}"
        )

    print("-" * 60)
    print("Total nodes:", graph.number_of_nodes())
    print("Total edges:", graph.number_of_edges())


In [19]:
def evaluate(actual, operator, expected):

    try:

        actual_num = float(actual)
        expected_num = float(expected)

        actual = actual_num
        expected = expected_num

    except:
        pass

    if operator in ["=", "=="]:
        return actual == expected

    if operator == "!=":
        return actual != expected

    if operator == ">":
        return actual > expected

    if operator == ">=":
        return actual >= expected

    if operator == "<":
        return actual < expected

    if operator == "<=":
        return actual <= expected

    return False



def symbolic_reasoning(data, knowledge_graph):

    results = []

    for rule in data.get("rules", []):

        rule_conditions = []
        rule_passed = True

        entities = list(dict.fromkeys(
            str(fact.get("entity", "Unknown"))
            for fact in data.get("facts", [])
        ))

        for condition in rule.get("conditions", []):

            attribute = condition["attribute"]
            operator = condition["operator"]
            expected = condition["value"]

            actual = None

            for entity in entities:

                actual = get_graph_fact(
                    knowledge_graph,
                    entity,
                    attribute
                )

                if actual is not None:
                    break

            if actual is None:

                status = "MISSING"
                rule_passed = False

            else:

                passed = evaluate(
                    actual,
                    operator,
                    expected
                )

                status = "PASS" if passed else "FAIL"

                if not passed:
                    rule_passed = False

            rule_conditions.append({
                "attribute": attribute,
                "operator": operator,
                "expected": expected,
                "actual": actual,
                "status": status
            })

        results.append({
            "conditions": rule_conditions,
            "status": (
                "SATISFIED"
                if rule_passed
                else "NOT SATISFIED"
            ),
            "conclusion": rule["conclusion"]
        })

    return results


In [20]:
def run_neuro_symbolic_system(user_input):

    print("=" * 75)
    print("       NEURO-SYMBOLIC LLM")
    print("       LOGIC-CONSTRAINED REASONING")
    print("=" * 75)

    print("\nUSER INPUT:")
    print(user_input)

    print("\n" + "-" * 75)
    print("1. NEURAL KNOWLEDGE EXTRACTION")
    print("-" * 75)

    raw_output = llm_extract(user_input)

    try:

        data = clean_llm_json(raw_output)
        data = normalize_knowledge(data)

        if not validate_knowledge(data):
            raise ValueError(
                "Extracted knowledge does not match the required structure."
            )

    except Exception as e:

        print("Unable to obtain structured knowledge.")
        print("Error:", e)
        return None

    print("\nFACTS:")

    for fact in data["facts"]:

        print(
            f"  {fact['entity']} → "
            f"{fact['attribute']} = "
            f"{fact['value']}"
        )

    print("\nRULES:")

    for i, rule in enumerate(
        data["rules"],
        1
    ):

        print(f"\n  Rule {i}:")

        for condition in rule["conditions"]:

            print(
                f"    {condition['attribute']} "
                f"{condition['operator']} "
                f"{condition['value']}"
            )

        conclusion = rule["conclusion"]

        print(
            f"    → {conclusion['attribute']} = "
            f"{conclusion['value']}"
        )

    print("\n" + "-" * 75)
    print("2. SYMBOLIC REASONING")
    print("-" * 75)


    knowledge_graph = build_knowledge_graph(data)
    print_knowledge_graph(knowledge_graph)

    reasoning = symbolic_reasoning(data, knowledge_graph)

    for i, result in enumerate(
        reasoning,
        1
    ):

        print(f"\nRule {i}:")

        for condition in result["conditions"]:

            if condition["status"] == "PASS":

                print(
                    f"  ✓ {condition['attribute']} "
                    f"{condition['operator']} "
                    f"{condition['expected']} "
                    f"(actual = {condition['actual']})"
                )

            elif condition["status"] == "FAIL":

                print(
                    f"  ✗ {condition['attribute']} "
                    f"{condition['operator']} "
                    f"{condition['expected']} "
                    f"(actual = {condition['actual']})"
                )

            else:

                print(
                    f"  ? {condition['attribute']} "
                    f"→ MISSING"
                )

        print(
            f"  Rule → {result['status']}"
        )

    print("\n" + "-" * 75)
    print("3. LOGIC-CONSTRAINT VERIFICATION")
    print("-" * 75)

    satisfied = [
        r for r in reasoning
        if r["status"] == "SATISFIED"
    ]

    if satisfied:

        print(
            "✓ All logical constraints satisfied."
        )

    else:

        print(
            "✗ One or more constraints failed."
        )

    print("\n" + "-" * 75)
    print("4. FINAL DECISION")
    print("-" * 75)

    if satisfied:

        for result in satisfied:

            conclusion = result["conclusion"]

            print(
                f"{conclusion['attribute']} = "
                f"{conclusion['value']}"
            )

    else:

        print("Condition not satisfied.")

    print("\n" + "-" * 75)
    print("5. EXPLAINABLE REASONING")
    print("-" * 75)

    failed = []

    missing = []

    for result in reasoning:

        for condition in result["conditions"]:

            if condition["status"] == "FAIL":

                failed.append(condition)

            elif condition["status"] == "MISSING":

                missing.append(condition)

    if failed:

        print(
            "The decision was not satisfied because:"
        )

        for condition in failed:

            print(
                f"• {condition['attribute']} "
                f"{condition['operator']} "
                f"{condition['expected']} "
                f"(actual = {condition['actual']})"
            )

    elif missing:

        print(
            "The decision could not be determined because:"
        )

        for condition in missing:

            print(
                f"• {condition['attribute']} "
                f"information is missing"
            )

    elif satisfied:

        print(
            "All logical constraints were satisfied, "
            "therefore the conclusion was derived by "
            "the symbolic engine."
        )

    print("\n" + "=" * 75)

    return {
        "knowledge": data,
        "reasoning": reasoning
    }

In [21]:
def validate_knowledge(data):

    if not isinstance(data, dict):
        return False

    if not isinstance(data.get("facts"), list):
        return False

    if not isinstance(data.get("rules"), list):
        return False

    if len(data["facts"]) == 0:
        return False

    if len(data["rules"]) == 0:
        return False

    for fact in data["facts"]:

        if not all(
            key in fact
            for key in ["entity", "attribute", "value"]
        ):
            return False

    for rule in data["rules"]:

        if "conditions" not in rule:
            return False

        if "conclusion" not in rule:
            return False

        if not isinstance(
            rule["conditions"],
            list
        ):
            return False

        if len(rule["conditions"]) == 0:
            return False

        for condition in rule["conditions"]:

            if not all(
                key in condition
                for key in [
                    "attribute",
                    "operator",
                    "value"
                ]
            ):
                return False

            if condition["operator"] not in [
                "==",
                "!=",
                ">",
                ">=",
                "<",
                "<="
            ]:
                return False

        conclusion = rule["conclusion"]

        if not all(
            key in conclusion
            for key in ["attribute", "value"]
        ):
            return False

    return True

In [22]:

test_data = {
    "facts": [
        {"entity": "Arun", "attribute": "credit_score", "value": 750},
        {"entity": "Arun", "attribute": "monthly_income", "value": 60000},
        {"entity": "Arun", "attribute": "existing_loan", "value": False}
    ],
    "rules": [
        {
            "conditions": [
                {"attribute": "credit_score", "operator": ">=", "value": 700},
                {"attribute": "monthly_income", "operator": ">=", "value": 50000},
                {"attribute": "existing_loan", "operator": "==", "value": False}
            ],
            "conclusion": {
                "attribute": "eligible",
                "value": True
            }
        }
    ]
}

kg = build_knowledge_graph(test_data)
print_knowledge_graph(kg)

print("\nRetrieved facts:")
print("Credit score:", get_graph_fact(kg, "Arun", "credit_score"))
print("Monthly income:", get_graph_fact(kg, "Arun", "monthly_income"))
print("Existing loan:", get_graph_fact(kg, "Arun", "existing_loan"))



KNOWLEDGE GRAPH
------------------------------------------------------------
entity:Arun --[HAS_FACT]--> fact:0
entity:Arun --[HAS_FACT]--> fact:1
entity:Arun --[HAS_FACT]--> fact:2
fact:0 --[HAS_VALUE]--> value:0
fact:1 --[HAS_VALUE]--> value:1
fact:2 --[HAS_VALUE]--> value:2
rule:0 --[HAS_CONDITION]--> condition:0:0
rule:0 --[HAS_CONDITION]--> condition:0:1
rule:0 --[HAS_CONDITION]--> condition:0:2
rule:0 --[CONCLUDES]--> conclusion:0
------------------------------------------------------------
Total nodes: 12
Total edges: 10

Retrieved facts:
Credit score: 750
Monthly income: 60000
Existing loan: False


In [23]:
user_input = """
A customer can receive an instant loan approval if the customer
has a credit score of at least 700, has a monthly income of at least
50000, and has no existing loan.

Arun has a credit score of 750.
Arun has a monthly income of 60000.
Arun has no existing loan.

Can Arun receive instant loan approval?
"""

result = run_neuro_symbolic_system(user_input)

       NEURO-SYMBOLIC LLM
       LOGIC-CONSTRAINED REASONING

USER INPUT:

A customer can receive an instant loan approval if the customer
has a credit score of at least 700, has a monthly income of at least
50000, and has no existing loan.

Arun has a credit score of 750.
Arun has a monthly income of 60000.
Arun has no existing loan.

Can Arun receive instant loan approval?


---------------------------------------------------------------------------
1. NEURAL KNOWLEDGE EXTRACTION
---------------------------------------------------------------------------

FACTS:
  Arun → credit_score = 750
  Arun → monthly_income = 60000
  Arun → existing_loan = False

RULES:

  Rule 1:
    credit_score >= 700
    monthly_income >= 50000
    existing_loan == False
    → instant_loan_approval = True

---------------------------------------------------------------------------
2. SYMBOLIC REASONING
---------------------------------------------------------------------------

KNOWLEDGE GRAPH
-------------

In [25]:
user_input = """
A passenger can board the flight if the passenger has a ticket score
of at least 1, has a check-in score of at least 1, and has a passport
score of at least 1.

Arun has a ticket score of 1.
Arun has a check-in score of 1.
Arun has a passport score of 1.

Can Arun board the flight?
"""

result = run_neuro_symbolic_system(user_input)

       NEURO-SYMBOLIC LLM
       LOGIC-CONSTRAINED REASONING

USER INPUT:

A passenger can board the flight if the passenger has a ticket score
of at least 1, has a check-in score of at least 1, and has a passport
score of at least 1.

Arun has a ticket score of 1.
Arun has a check-in score of 1.
Arun has a passport score of 1.

Can Arun board the flight?


---------------------------------------------------------------------------
1. NEURAL KNOWLEDGE EXTRACTION
---------------------------------------------------------------------------

FACTS:
  Arun → ticket_score = 1
  Arun → check_in_score = 1
  Arun → passport_score = 1

RULES:

  Rule 1:
    ticket_score >= 1
    check_in_score >= 1
    passport_score >= 1
    → can_board_flight = True

---------------------------------------------------------------------------
2. SYMBOLIC REASONING
---------------------------------------------------------------------------

KNOWLEDGE GRAPH
--------------------------------------------------------

In [26]:
user_input = """
A patient can receive a priority appointment if the patient has a
priority score of at least 80, has a waiting time of at least 30 days,
and has a doctor rating of at least 4.

Meera has a priority score of 90.
Meera has a waiting time of 45 days.
Meera has a doctor rating of 4.5.

Can Meera receive a priority appointment?
"""

result = run_neuro_symbolic_system(user_input)

       NEURO-SYMBOLIC LLM
       LOGIC-CONSTRAINED REASONING

USER INPUT:

A patient can receive a priority appointment if the patient has a
priority score of at least 80, has a waiting time of at least 30 days,
and has a doctor rating of at least 4.

Meera has a priority score of 90.
Meera has a waiting time of 45 days.
Meera has a doctor rating of 4.5.

Can Meera receive a priority appointment?


---------------------------------------------------------------------------
1. NEURAL KNOWLEDGE EXTRACTION
---------------------------------------------------------------------------

FACTS:
  Meera → priority_score = 90
  Meera → waiting_time = 45
  Meera → doctor_rating = 4.5

RULES:

  Rule 1:
    priority_score >= 80
    waiting_time >= 30
    doctor_rating >= 4
    → can_receive_priority_appointment = True

---------------------------------------------------------------------------
2. SYMBOLIC REASONING
---------------------------------------------------------------------------

KNOWLED

In [24]:
user_input = """
A customer can receive a home loan if the customer has a credit score
of at least 700, has a monthly income of at least 60000,
and has savings of at least 500000.

Arun has a credit score of 750.
Arun has a monthly income of 70000.
Arun has savings of 300000.

Can Arun receive the home loan?
"""

result = run_neuro_symbolic_system(user_input)

       NEURO-SYMBOLIC LLM
       LOGIC-CONSTRAINED REASONING

USER INPUT:

A customer can receive a home loan if the customer has a credit score
of at least 700, has a monthly income of at least 60000,
and has savings of at least 500000.

Arun has a credit score of 750.
Arun has a monthly income of 70000.
Arun has savings of 300000.

Can Arun receive the home loan?


---------------------------------------------------------------------------
1. NEURAL KNOWLEDGE EXTRACTION
---------------------------------------------------------------------------

FACTS:
  Arun → credit_score = 750
  Arun → monthly_income = 70000
  Arun → savings = 300000

RULES:

  Rule 1:
    credit_score >= 700
    monthly_income >= 60000
    savings >= 500000
    → eligible = True

---------------------------------------------------------------------------
2. SYMBOLIC REASONING
---------------------------------------------------------------------------

KNOWLEDGE GRAPH
----------------------------------------------

In [27]:
user_input = """
A customer is eligible for a business loan if the customer has a credit score
of at least 700, has a monthly income of at least 80000,
and has savings of at least 500000.

Ravi has a credit score of 580.
Ravi has a monthly income of 40000.
Ravi has savings of 200000.

Is Ravi eligible for the business loan?
"""

result = run_neuro_symbolic_system(user_input)

       NEURO-SYMBOLIC LLM
       LOGIC-CONSTRAINED REASONING

USER INPUT:

A customer is eligible for a business loan if the customer has a credit score
of at least 700, has a monthly income of at least 80000,
and has savings of at least 500000.

Ravi has a credit score of 580.
Ravi has a monthly income of 40000.
Ravi has savings of 200000.

Is Ravi eligible for the business loan?


---------------------------------------------------------------------------
1. NEURAL KNOWLEDGE EXTRACTION
---------------------------------------------------------------------------

FACTS:
  Ravi → credit_score = 580
  Ravi → monthly_income = 40000
  Ravi → savings = 200000

RULES:

  Rule 1:
    credit_score >= 700
    monthly_income >= 80000
    savings >= 500000
    → eligible = True

---------------------------------------------------------------------------
2. SYMBOLIC REASONING
---------------------------------------------------------------------------

KNOWLEDGE GRAPH
------------------------------